# Import Libraries
This section imports the necessary libraries for data manipulation and environment variable management.

In [7]:
import pandas as pd
import json
from pandas import read_csv
from dotenv import load_dotenv

# Read CSV File
This section reads a CSV file into a Pandas DataFrame and displays the first few rows.

In [8]:
# Read the CSV file into a DataFrame
df = pd.read_csv('o1-vs-4o-scenarios-cn.csv')
df.head()

,Sector,Use Case,High Level,Scenario,Prompt,o1,gpt4o,Overview,simple_comparison,complex_comparison,o1_time,gpt4o_time
0,Banking,Credit Risk Assessment and Management,NaN,NaN,<purpose>Credit Risk Assessment</purpose>\n<in...,**Credit Risk Assessment**\n\n**1. Assign Rati...,### Credit Risk Assessment\n\n**1. Assign a Ra...,"John Doe is applying for a $200,000 mortgage o...",Response 2 is more comprehensive than Response...,**Comparison of the Two Responses:**\n\n**1. H...,40.26,7.24
1,Banking,Fraud Detection and Prevention,NaN,Hypothetical Example: O1 detects fraud by anal...,<purpose>Fraud Detection in Banking Transactio...,**Fraud Detection Analysis for Account 1234567...,**Fraud Detection Report for John Doe (Account...,A real-time system is analyzing transaction hi...,Response 2 offers a more detailed analysis by ...,**Comparison of Responses: Fraud Detection Ana...,40.21,8.87
2,Banking,Regulatory Compliance and Reporting,NaN,Hypothetical Example: O1 automates regulatory ...,<purpose>Automated AML Compliance Evaluation</...,**AML Compliance Evaluation**\n\n**Flagged Tra...,### Compliance Evaluation\n\n#### Transaction ...,The system is tasked with ensuring regulatory ...,Response 2 is superior because it correctly id...,**Comparison of Responses:**\n\nBoth responses...,40.21,8.87
3,Banking,Customer Relationship Management,NaN,Hypothetical Example: O1 helps banks personali...,<purpose>Customer Behavior Analysis and Person...,**Detailed Client Analysis and Personalized Re...,# Customer Behavior Analysis and Personalized ...,The bank wants to enhance loyalty for its top ...,Response 1 offers a more comprehensive and str...,**Comparison of Responses:**\n\n**Overall Stru...,40.21,8.87
4,Banking,Investment and Portfolio Management,NaN,Hypothetical Example: O1 analyzes market trend...,<purpose>Investment Portfolio Optimization</pu...,**Investment Portfolio Optimization Report**\n...,### Investment Portfolio Optimization\n\n#### ...,A client seeks guidance on how to allocate a $...,"Response 2 is more comprehensive, offering an ...",**Comparison of the Two Responses:**\n\nBoth r...,40.21,8.87


# Remove First Line
This section removes the first line of the DataFrame, which may contain headers or unwanted data.

In [9]:
# Remove the first line of the DataFrame
# df = df.iloc[1:]
# df.head()

# Azure OpenAI Chat Completion
This section sends each JSON object to the Azure OpenAI endpoint to generate a response using the chat completion API.

In [10]:
import os
import requests
# Define the Azure OpenAI endpoint and API key
load_dotenv()

# Read Azure OpenAI credentials from environment variables
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")


# Columns to translate
columns_to_translate = ["o1", "gpt4o", "Overview", "simple_comparison", "complex_comparison"]

# # Function to translate text to Chinese using GPT-4
# def translate_to_chinese(text):
#     if pd.isna(text) or text.strip() == '':
#         return text  # Return the text as is if it's NaN or empty
#     try:
#         response = openai.ChatCompletion.create(
#             model='gpt-4',
#             messages=[
#                 {"role": "system", "content": "You are a translator that converts English text to Chinese and keep the original format"},
#                 {"role": "user", "content": f"Please translate the following text to Chinese:\n\n{text}"}
#             ],
#             max_tokens=600,
#             n=1,
#             temperature=0.3,
#         )
#         translated_text = response.choices[0].message['content'].strip()
#         return translated_text
#     except Exception as e:
#         print(f"Error translating text: {e}")
#         return text  # Return the original text if there's an error

       

def generate_content_azure(system_prompt, user_input, temperature=0.3, top_p=0.9, max_tokens=2000):
    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_DEPLOYMENT_NAME}/chat/completions?api-version=2024-08-01-preview"
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY
    }
    payload = {
        "messages": [
            {"role": "system", "content": f"{system_prompt}"},
            {"role": "user", "content": f"{user_input}"}
        ],
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        # "response_format": {
        #     "type": "json_object"
        # }
    }

    try:
        # print(payload)
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip()
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        print(f"Response content: {response.content if response else 'No response'}")
        raise

SYSTEM_PROMPT = "You are a translator that converts following English text to Chinese and keep the original format:\n\n"

# 2. Translate and replace the content in the specified fields
for column in columns_to_translate:
    if column in df.columns:
       for index, value in df[column].items():
            if pd.isna(value) or str(value).strip() == '':
                continue  # Skip empty or NaN values
            # Translate the value
            # print(value)
            translated_output = generate_content_azure(
                SYSTEM_PROMPT,
                value,
                temperature=0.4,
                max_tokens=1000
            )
            print(translated_output)
            # Replace the original value with the translated output
            df.at[index, column] = translated_output 
    else:
        print(f"Column '{column}' does not exist in the DataFrame.")
 


**信用风险评估**

**1. 为每个标准分配评级：**

- **信用评分：** **A**
  - 客户的信用评分为**720**，属于**A（700或以上）**。

- **债务收入比：** **B**
  - 客户的债务收入比为**35%**，属于**B（30-39%）**。

- **贷款价值比：** **C**
  - 申请的贷款金额为**$200,000**，房屋评估价值为**$250,000**。
  - 贷款价值比 = （贷款金额 / 评估价值）× 100 = （200,000 / 250,000）× 100 = **80%**，属于**C（80-89%）**。

- **就业状况：** **A**
  - 客户为**全职**就业，且就业历史**稳定**，属于**A（全职，稳定）**。

- **市场趋势：** **B**
  - 市场被描述为**稳定，需求适中，库存低**，属于**B（稳定，需求适中，库存低）**。

**综合评级：** **A B C A B**

---

**2. 查找综合评级“ABCAB”的政策限制：**

提供的政策表中没有包含综合评级**“ABCAB”**。然而，类似的评级可以提供指导：

- **AAABC**：最高贷款金额为**$300,000**，最低利率为**4.25%**。
- **AAABB**：最高贷款金额为**$350,000**，最低利率为**4%**。

鉴于客户的综合评级为**“ABCAB”**，由于第三位的**C**，评级略低，因此我们可以做出以下**假设**：

- **估计最高贷款金额：** **$300,000**。
- **估计最低利率：** **4.25%**。

---

**3. 比较申请的贷款金额和提供的利率与政策限制：**

- **申请的贷款金额：** **$200,000**（低于估计的最高**$300,000**）。
- **提供的利率：** **4%**（低于估计的最低**4.25%**）。

---

**4. 决策和条件：**

**决策：** **有条件批准**

**在最终批准前需验证的条件：**

1. **政策确认：**
   - 根据银行的信用风险评估政策，验证综合评级**“ABCAB”**的确切政策限制。

2. **利率调整：**
   - 由于提供

KeyboardInterrupt: 

In [11]:
# df.to_csv('translated.csv', index=False, encoding='utf-8')
df.head()

,Sector,Use Case,High Level,Scenario,Prompt,o1,gpt4o,Overview,simple_comparison,complex_comparison,o1_time,gpt4o_time
0,Banking,Credit Risk Assessment and Management,NaN,NaN,<purpose>Credit Risk Assessment</purpose>\n<in...,**信用风险评估**\n\n**1. 为每个标准分配评级：**\n\n- **信用评分：**...,### Credit Risk Assessment\n\n**1. Assign a Ra...,"John Doe is applying for a $200,000 mortgage o...",Response 2 is more comprehensive than Response...,**Comparison of the Two Responses:**\n\n**1. H...,40.26,7.24
1,Banking,Fraud Detection and Prevention,NaN,Hypothetical Example: O1 detects fraud by anal...,<purpose>Fraud Detection in Banking Transactio...,**Fraud Detection Analysis for Account 1234567...,**Fraud Detection Report for John Doe (Account...,A real-time system is analyzing transaction hi...,Response 2 offers a more detailed analysis by ...,**Comparison of Responses: Fraud Detection Ana...,40.21,8.87
2,Banking,Regulatory Compliance and Reporting,NaN,Hypothetical Example: O1 automates regulatory ...,<purpose>Automated AML Compliance Evaluation</...,**AML Compliance Evaluation**\n\n**Flagged Tra...,### Compliance Evaluation\n\n#### Transaction ...,The system is tasked with ensuring regulatory ...,Response 2 is superior because it correctly id...,**Comparison of Responses:**\n\nBoth responses...,40.21,8.87
3,Banking,Customer Relationship Management,NaN,Hypothetical Example: O1 helps banks personali...,<purpose>Customer Behavior Analysis and Person...,**Detailed Client Analysis and Personalized Re...,# Customer Behavior Analysis and Personalized ...,The bank wants to enhance loyalty for its top ...,Response 1 offers a more comprehensive and str...,**Comparison of Responses:**\n\n**Overall Stru...,40.21,8.87
4,Banking,Investment and Portfolio Management,NaN,Hypothetical Example: O1 analyzes market trend...,<purpose>Investment Portfolio Optimization</pu...,**Investment Portfolio Optimization Report**\n...,### Investment Portfolio Optimization\n\n#### ...,A client seeks guidance on how to allocate a $...,"Response 2 is more comprehensive, offering an ...",**Comparison of the Two Responses:**\n\nBoth r...,40.21,8.87
